In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_anim
from IPython.display import HTML
from qutip import *

# Setup Simulation (Leaky Cavity)
N = 15
w = 1.0
kappa = 0.1  # Strong decay 
a = destroy(N)
H = w * a.dag() * a

# Initial State: Fock State |1> (1 Photon)
psi0 = basis(N, 1)

# Collapse Operator (The Leak)
c_ops = [np.sqrt(kappa) * a]

# Time Evolution
t_max = 30
frames = 60
tlist = np.linspace(0, t_max, frames)

print("1. Running Decay Simulation")
# We use 'mesolve' with c_ops
output = mesolve(H, psi0, tlist, c_ops, [])

# Pre-Calculate Wigner Data
print("2. Calculating Wigner frames")
xvec = np.linspace(-5, 5, 100)
wigner_data = []

for state in output.states:
    # No ptrace needed because there is no atom here, just the cavity
    wigner_data.append(wigner(state, xvec, xvec))

# Build Animation
fig, ax = plt.subplots(figsize=(6, 6))

# Fixed color scale
w_max = np.max(wigner_data[0])
w_min = -w_max

mesh = ax.contourf(xvec, xvec, wigner_data[0], 100, cmap="RdBu_r", levels=np.linspace(w_min, w_max, 100))
ax.set_aspect('equal')
ax.set_title("Photon Decay")

def update(frame):
    ax.clear()
    ax.contourf(xvec, xvec, wigner_data[frame], 100, cmap="RdBu_r", levels=np.linspace(w_min, w_max, 100))
    ax.set_title(f"Dissipation: Photon Leakage (t={tlist[frame]:.1f})")
    ax.set_xlabel("Position q")
    ax.set_ylabel("Momentum p")
    return ax,

anim = mpl_anim.FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_anim
from IPython.display import HTML
from qutip import *

# --- 1. Define JCH Hamiltonian (2 Sites) ---
# N=3 is enough because we only have 1 excitation
N = 3
w = 1.0
g = 0.05
J = 0.05 

# Operators for 4 subsystems: [CavA, AtomA, CavB, AtomB]
a1  = tensor(destroy(N), qeye(2), qeye(N), qeye(2))
sm1 = tensor(qeye(N), sigmam(), qeye(N), qeye(2))
a2  = tensor(qeye(N), qeye(2), destroy(N), qeye(2))
sm2 = tensor(qeye(N), qeye(2), qeye(N), sigmam())

# Hamiltonian
H_local = w*a1.dag()*a1 + w*a2.dag()*a2 + \
          0.5*w*(sz1:=tensor(qeye(N), sigmaz(), qeye(N), qeye(2))) + \
          0.5*w*(sz2:=tensor(qeye(N), qeye(2), qeye(N), sigmaz())) + \
          g*(a1.dag()*sm1 + a1*sm1.dag()) + \
          g*(a2.dag()*sm2 + a2*sm2.dag())

H_hop = -J * (a1.dag()*a2 + a1*a2.dag())
H = H_local + H_hop

# Initial State: Atom A Excited, everything else Ground/Vacuum
psi0 = tensor(basis(N,0), basis(2,0), basis(N,0), basis(2,1))

# Evolution
t_max = np.pi / J  # Approximate transfer time
frames = 100
tlist = np.linspace(0, t_max, frames)

# Observables: Population of [Atom A, Cavity A, Cavity B, Atom B]
# Note: For atoms, we project onto excited state basis(2,0)
pop_AtomA = tensor(qeye(N), basis(2,0)*basis(2,0).dag(), qeye(N), qeye(2))
pop_CavA  = a1.dag() * a1
pop_CavB  = a2.dag() * a2
pop_AtomB = tensor(qeye(N), qeye(2), qeye(N), basis(2,0)*basis(2,0).dag())

output = mesolve(H, psi0, tlist, [], [pop_AtomA, pop_CavA, pop_CavB, pop_AtomB])

# --- 2. Animate Bar Chart ---
fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Atom A', 'Cavity A', 'Cavity B', 'Atom B']
colors = ['red', 'blue', 'blue', 'purple']

# Initialize Bars
bars = ax.bar(labels, [1, 0, 0, 0], color=colors)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Excitation Probability")
title = ax.set_title("Quantum State Transfer: t=0.0")

def update(frame):
    # Get new heights for this frame
    heights = [output.expect[0][frame], 
               output.expect[1][frame], 
               output.expect[2][frame], 
               output.expect[3][frame]]
    
    # Update each bar
    for bar, h in zip(bars, heights):
        bar.set_height(h)
        
    title.set_text(f"Quantum State Transfer: t={tlist[frame]:.2f}")
    return bars

anim = mpl_anim.FuncAnimation(fig, update, frames=frames, interval=50, blit=False)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import *
import matplotlib.animation as mpl_anim
from IPython.display import HTML

# --- 1. Simulation (Single Rabi Flop) ---
# Start in Ground State, drive it to Excited and back
psi0 = basis(2, 0) 
H = 0.5 * sigmax() # Hamiltonian that causes rotation around X-axis
tlist = np.linspace(0, 2*np.pi, 60)

# Run solver returning STATES (empty list [] for observables)
output = mesolve(H, psi0, tlist, [], [])

# --- 2. Calculate Vectors Manually ---
vecs = []
for state in output.states:
    # Calculate <x>, <y>, <z> for each time step
    vecs.append([expect(sigmax(), state), expect(sigmay(), state), expect(sigmaz(), state)])
vecs = np.array(vecs) # Shape (frames, 3)

# --- 3. Bloch Sphere Animation ---
fig = plt.figure(figsize=(6, 6))
axes = fig.add_subplot(111, projection='3d')
b = Bloch(fig=fig, axes=axes)

# Add the initial vector
b.vector_color = ['r']
b.add_vectors([vecs[0][0], vecs[0][1], vecs[0][2]])

def update(frame):
    b.clear() # Clear old vectors
    # Add new vector corresponding to the current frame
    b.add_vectors([vecs[frame][0], vecs[frame][1], vecs[frame][2]])
    b.make_sphere() 
    return axes,

# Create Animation
anim = mpl_anim.FuncAnimation(fig, update, frames=len(vecs), interval=50, blit=False)

plt.close()
HTML(anim.to_jshtml())